### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [3]:
import os
from dotenv import load_dotenv
from langfuse.langchain import CallbackHandler
load_dotenv()
from langchain.chat_models import init_chat_model

langfuse_trace = CallbackHandler()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# model = init_chat_model("groq:qwen/qwen3-32b")
model = init_chat_model("granite4", model_provider="ollama")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content="Parrots have a natural inclination to mimic and communicate using sounds, including speech. There are several reasons why they talk:\n\n1. Social interaction: Parrots are highly social animals that thrive on interaction with their flock or humans. Talking is one way for them to engage in this communication.\n\n2. Mimicry instinct: Like other birds, parrots have a strong innate ability to mimic sounds and speech from their environment. This trait has been honed through evolution as a means of survival.\n\n3. Bonding with humans: When kept as pets, parrots often bond closely with their human caregivers. The caregiver's voice becomes an important part of the parrot's social life, leading them to imitate it.\n\n4. Learning and repetition: Parrots learn by listening and repeating sounds they hear frequently. If a parrot hears its name or words spoken consistently by its owner, it will likely mimic those sounds over time.\n\n5. Cognitive abilities: Many species of parrots 

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a given location."""
    return f"It is a sunny in {location}"

In [4]:
model_with_tools = model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?", config={
    "callbacks": [langfuse_trace]
})
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={} response_metadata={'model': 'granite4', 'created_at': '2026-06-23T14:15:22.437822638Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3733980748, 'load_duration': 348267891, 'prompt_eval_count': 173, 'prompt_eval_duration': 1590879000, 'eval_count': 27, 'eval_duration': 1779712000, 'logprobs': None, 'model_name': 'granite4', 'model_provider': 'ollama'} id='lc_run--019ef4d5-e3ad-79f2-8099-16719964354c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'f7da6f9a-c87f-4a88-af81-25b410c12a20', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 173, 'output_tokens': 27, 'total_tokens': 200}
Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [8]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages, config={"callbacks": [langfuse_trace]})
print("ai_msg: ", ai_msg)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    print("tool_call: ", tool_call)
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

print("messages: ", messages)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages, config={"callbacks": [langfuse_trace]})
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

ai_msg:  content='' additional_kwargs={} response_metadata={'model': 'granite4', 'created_at': '2026-06-23T14:21:24.666112601Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2161985661, 'load_duration': 418004995, 'prompt_eval_count': 172, 'prompt_eval_duration': 63587000, 'eval_count': 27, 'eval_duration': 1677026000, 'logprobs': None, 'model_name': 'granite4', 'model_provider': 'ollama'} id='lc_run--019ef4db-70c4-7c42-88b6-9ebc119a6388-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'd6c37973-18d9-4e9a-9958-7536efd81096', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 172, 'output_tokens': 27, 'total_tokens': 199}
tool_call:  {'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'd6c37973-18d9-4e9a-9958-7536efd81096', 'type': 'tool_call'}
messages:  [{'role': 'user', 'content': "What's the weather in Boston?"}, AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'granite4', 'created_at': '

In [9]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'granite4', 'created_at': '2026-06-23T14:21:24.666112601Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2161985661, 'load_duration': 418004995, 'prompt_eval_count': 172, 'prompt_eval_duration': 63587000, 'eval_count': 27, 'eval_duration': 1677026000, 'logprobs': None, 'model_name': 'granite4', 'model_provider': 'ollama'}, id='lc_run--019ef4db-70c4-7c42-88b6-9ebc119a6388-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'd6c37973-18d9-4e9a-9958-7536efd81096', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 172, 'output_tokens': 27, 'total_tokens': 199}),
 ToolMessage(content="It's sunny in Boston", name='get_weather', tool_call_id='d6c37973-18d9-4e9a-9958-7536efd81096')]